## Paso 0: Importar librerías

Cargamos todas las librerías desde el principio. Si algún import falla, mejor saberlo aquí que en el paso 9.

In [ ]:
# Manipulación de datos
import pandas as pd
import numpy as np
from scipy import stats

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocesamiento
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    OneHotEncoder, LabelEncoder
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Modelos
from sklearn.linear_model import (
    LinearRegression, LogisticRegression,
    Ridge, Lasso, ElasticNet
)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    GradientBoostingClassifier, GradientBoostingRegressor
)

# Métricas de regresión
from sklearn.metrics import (
    mean_squared_error, root_mean_squared_error,
    mean_absolute_error, r2_score
)

# Métricas de clasificación
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_recall_curve
)

import warnings
warnings.filterwarnings('ignore')
pd.options.mode.copy_on_write = True

## Paso 1: Cargar datos

Traemos los datos al entorno. En esta etapa solo cargamos y echamos un vistazo rápido para hacernos una idea de con qué trabajamos.

In [ ]:
df = pd.read_csv("ponga_su_ruta.csv")

print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()
df.describe()

## Paso 2: Definición del problema

Antes de tocar los datos, debemos saber qué queremos resolver:

- **¿Predecir un valor numérico continuo?** → Regresión
- **¿Clasificar en categorías?** → Clasificación

Esto determina los modelos, las métricas y cómo interpretamos los resultados.

> ⚠️ Un target numérico no implica siempre regresión. Si los valores son códigos de categoría (0, 1, 2 representando clases), es clasificación. Comprueba siempre el significado del dato, no solo su tipo.

In [ ]:
TARGET = 'y'  # ← Cambia esto por el nombre de tu columna objetivo

n_unique = df[TARGET].nunique()
print(f"Valores únicos en la target: {n_unique}")

if n_unique <= 20:
    print("→ Posible CLASIFICACIÓN")
    print(df[TARGET].value_counts())
else:
    print("→ Posible REGRESIÓN")
    print(df[TARGET].describe())

## Paso 3: División de Train y Test

Antes de explorar o modelar, separamos una parte de los datos que el modelo **nunca verá durante el entrenamiento**.

| Conjunto | Proporción habitual | Uso |
|---|---|---|
| **Train** | 70–80 % | Entrenar y ajustar el modelo |
| **Test** | 20–30 % | Evaluación final, solo una vez |

### ⚠️ El error más común en ML: Data Leakage

**Data leakage** (fuga de datos) significa que el modelo recibe, de forma indirecta, información sobre el test antes de la evaluación. Hace que las métricas sean artificialmente optimistas y que el modelo falle más de lo esperado en producción.

El caso más común: calcular media/desviación para el escalado **usando todo el dataset** antes del split.

```
❌ INCORRECTO
    dataset completo
         │
    fit scaler  ← usa datos de test sin querer
         │
    split train / test

✅ CORRECTO
    dataset completo
         │
    split train / test
    ┌────┴────┐
  train      test
    │          │
  fit +     transform
  transform   (solo)
```

**Regla de oro: `fit()` solo sobre train. `transform()` sobre train y test por separado.**

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20 % para test
    random_state=42,     # Fija la semilla → resultados reproducibles
    # stratify=y         # Descomentar en clasificación con clases desbalanceadas
)

print(f"Train: {X_train.shape[0]} filas  ({X_train.shape[0]/len(df)*100:.0f} %)")
print(f"Test:  {X_test.shape[0]} filas  ({X_test.shape[0]/len(df)*100:.0f} %)")

> 💡 **`random_state=42`**: sin esto, cada ejecución genera un split distinto y los resultados son irreproducibles. Cualquier número funciona; 42 es convención.

> 💡 **`stratify=y`**: garantiza que la proporción de clases sea la misma en train y test. Fundamental cuando las clases están desbalanceadas.

## Paso 4: Limpieza de los datos

Revisamos problemas fundamentales: tipos de datos incorrectos, valores faltantes evidentes, duplicados.

Las transformaciones avanzadas (escalado, encodings, normalización de distribuciones) se hacen en el Paso 6, **siempre partiendo de X_train**.

> ⚠️ Trabajar sobre `X_train` desde ya, no sobre `df`. Cualquier estadístico que calculemos (media, mediana…) debe calcularse solo con datos de train.

In [ ]:
# Tipos de datos
print(X_train.dtypes)
print()

# Valores faltantes
missing = X_train.isna().sum()
missing_pct = (missing / len(X_train) * 100).round(1)
resumen = pd.DataFrame({'count': missing, '%': missing_pct})
print(resumen[resumen['count'] > 0].sort_values('%', ascending=False))
print()

# Duplicados
print(f"Duplicados: {X_train.duplicated().sum()}")

## Paso 5: Comprensión de variables — mini-EDA

El objetivo del EDA no es hacer un análisis exhaustivo: es detectar los problemas y patrones que van a condicionar el modelado.

### 5.1 Pairplot inicial

Visión rápida de múltiples variables a la vez. Útil para detectar relaciones lineales evidentes, clusters y outliers flagrantes. Evítalo con más de 8–10 features continuas, se vuelve ilegible.

In [ ]:
# Seleccionamos las features numéricas y añadimos la target
cols_plot = X_train.select_dtypes('number').columns.tolist()[:6]  # max 6 para legibilidad

sns.pairplot(
    X_train[cols_plot].join(y_train),
    diag_kind='kde',
    plot_kws={'alpha': 0.4, 's': 10}
)
plt.suptitle('Pairplot de features numéricas', y=1.02)
plt.show()

### 5.2 Estudio de la target

Analiza tu variable objetivo antes que cualquier feature. Su distribución y balance condicionan métricas y modelos.

- En **regresión**: busca asimetría (skewness), outliers extremos. Si el skew es alto, puede convenir transformar la target también.
- En **clasificación**: revisa el desbalance de clases. Si una clase tiene < 10 % de las muestras, accuracy no sirve como métrica principal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma con curva de densidad
sns.histplot(y_train, kde=True, ax=axes[0])
axes[0].set_title(f"Distribución de '{TARGET}'  |  skew: {y_train.skew():.2f}")

# Boxplot para ver outliers
axes[1].boxplot(y_train.dropna(), vert=False)
axes[1].set_title(f"Boxplot de '{TARGET}'")
axes[1].set_xlabel(TARGET)

plt.tight_layout()
plt.show()

# Para clasificación — descomenta esto:
# sns.countplot(x=y_train)
# plt.title('Distribución de clases')
# print(y_train.value_counts(normalize=True).mul(100).round(1))

### 5.3 Análisis de correlaciones

Revisamos relaciones lineales entre variables.

- **Correlación alta entre una feature y la target**: buen indicador de predictibilidad.
- **Correlación alta entre dos features**: señal de multicolinealidad.

> ⚠️ **Multicolinealidad**: si `superficie_m2` y `superficie_ft2` explican exactamente lo mismo, el modelo lineal no puede separar el efecto de cada una y los coeficientes serán inestables. Solución: eliminar una de las dos, o usar regularización L1/L2.

> 💡 **Pearson vs Spearman**: `.corr()` usa Pearson por defecto, que detecta solo relaciones *lineales*. Si en los scatterplots ves relaciones curvilíneas claras, usa `.corr(method='spearman')` que es invariante a transformaciones monótonas.

In [ ]:
corr = X_train.join(y_train).corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', vmin=-1, vmax=1,
            cmap='coolwarm', linewidths=0.5, ax=ax)
ax.set_title('Matriz de correlaciones (Pearson) — calculada sobre train')
plt.tight_layout()
plt.show()

# Correlación de cada feature con la target, ordenada
print("\nCorrelación con la target:")
print(corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).to_string())

### 5.4 Análisis univariante de features

Observa cada feature individualmente: distribución, asimetría, outliers. Esto te dice qué tratamientos van a necesitar.

In [ ]:
X_train.select_dtypes('number').hist(figsize=(14, 10), bins=30, edgecolor='white')
plt.suptitle('Distribución de features numéricas', y=1.01)
plt.tight_layout()
plt.show()

# Skewness de cada feature numérica
skew = X_train.select_dtypes('number').skew().sort_values(key=abs, ascending=False)
print("Skewness por feature:")
print(skew.to_string())

### 5.5 Análisis bivariante

Compara cada feature con la target para ver qué relaciones existen.

In [ ]:
feature = 'feature1'  # ← Cambiar por la columna que quieras explorar

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(X_train[feature], y_train, alpha=0.4, s=15)
ax.set_xlabel(feature)
ax.set_ylabel(TARGET)
ax.set_title(f'{feature} vs {TARGET}')
plt.tight_layout()
plt.show()

# Para clasificación: boxplot por clase
# sns.boxplot(x=y_train, y=X_train[feature])

## Paso 6: Tratamiento de variables

Preparamos los datos para que los modelos puedan procesarlos. Todo lo que aprendemos en este paso (medias, escalas, encoders…) se **calcula sobre train** y se **aplica igual a test**.

### 6.1 Eliminación de features

Tras el EDA, evaluamos si sobran variables. Candidatas a eliminar:

- Valor constante o casi constante (varianza ≈ 0).
- Muchos missings (> 30–40 %, depende del dataset y su relevancia).
- Identificadores (IDs, nombres, índices únicos por fila).
- Alta cardinalidad sin tratamiento especial (ej: 10.000 ciudades distintas).
- Multicolinealidad severa con otra feature más informativa.
- Strings largos sin procesamiento NLP.

In [ ]:
# Ver varianza de cada feature numérica — las de varianza muy baja son sospechosas
print(X_train.var().sort_values().head(10))

# Eliminar manualmente
# X_train = X_train.drop(columns=['id', 'nombre', 'feature_redundante'])
# X_test  = X_test.drop(columns=['id', 'nombre', 'feature_redundante'])

### 6.2 Duplicados

Los duplicados pueden sesgar el entrenamiento (el modelo ve esas filas más veces). Antes de eliminarlos, entiende qué define una fila única en tu problema.

In [ ]:
print(f"Duplicados: {X_train.duplicated().sum()}")

# Eliminar duplicados — siempre con la misma máscara en X e y
mask_dup = X_train.duplicated(keep='first')
X_train = X_train[~mask_dup].reset_index(drop=True)
y_train = y_train[~mask_dup].reset_index(drop=True)  # ← mismo índice que X_train

# Por una columna concreta
# mask = X_train.duplicated(subset=['cliente_id'], keep='last')

### 6.3 Missings (valores faltantes)

Un missing es un valor ausente (`NaN`). No es un 0 ni un string vacío. La mayoría de modelos no los toleran.

| Estrategia | Cuándo usarla |
|---|---|
| Eliminar filas | Pocos missings (< 5 %) |
| Eliminar columna | > 40 % de missings y poca relevancia |
| Imputar con media | Distribución simétrica, sin outliers |
| Imputar con mediana | Distribución sesgada o con outliers (preferida en general) |
| Imputar con moda | Variables categóricas |
| Imputar con valor fijo | Cuando el missing tiene un significado propio |
| Flag + imputación | Cuando el hecho de que falte es informativo en sí mismo |

In [ ]:
# Porcentaje de missings por columna
missing_pct = X_train.isna().mean().mul(100).round(1)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

# Imputación con mediana usando sklearn (fit en train, transform en ambos)
cols_num = X_train.select_dtypes('number').columns.tolist()

imputer = SimpleImputer(strategy='median')
X_train[cols_num] = imputer.fit_transform(X_train[cols_num])  # fit + transform
X_test[cols_num]  = imputer.transform(X_test[cols_num])        # solo transform

# Flag de missing (útil cuando el hecho de que falte aporta info)
# col = 'ingresos'
# X_train[f'{col}_missing'] = X_train[col].isna().astype(int)
# X_test[f'{col}_missing']  = X_test[col].isna().astype(int)

### 6.4 Anomalías y errores

No son outliers estadísticos: son datos **claramente incorrectos**.
Edades negativas, fechas en el año 9999, precios de 0 donde no tiene sentido.

Se tratan como missings (se ponen a NaN) y luego se imputan, o se corrigen manualmente.

In [ ]:
# Detectar rangos imposibles
print(X_train.describe().T[['min', 'max']])

# Ejemplo: edades negativas → reemplazar por NaN y reimputar
# X_train.loc[X_train['edad'] < 0, 'edad'] = np.nan
# X_test.loc[X_test['edad'] < 0, 'edad']   = np.nan

### 6.5 Outliers

Los outliers son valores atípicos estadísticamente, pero no necesariamente erróneos. Un sueldo de 500.000 € puede ser real (un directivo), no un error.

**¿Cuándo afectan?**
- Modelos lineales, redes neuronales, KNN: bastante sensibles.
- Árboles de decisión y Random Forest: robustos, los outliers suelen no ser problema.

**Detección:**
- **IQR**: límites = Q1 − 1.5·IQR  y  Q3 + 1.5·IQR. Lo que queda fuera es outlier.
- **Z-score**: valores a más de 2–3 desviaciones estándar de la media.

**Tratamiento:**
- Eliminar (si son pocos y claramente erróneos) — siempre con la misma máscara en X e y.
- **Winsorización**: caparlo al límite sin eliminar la fila.
- Transformación logarítmica (comprime la cola).
- No hacer nada si el modelo es robusto (árboles).

In [ ]:
col = 'feature'  # ← Cambiar por la columna a analizar

Q1 = X_train[col].quantile(0.25)
Q3 = X_train[col].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

n_out = ((X_train[col] < lower) | (X_train[col] > upper)).sum()
print(f'Outliers en "{col}": {n_out}  ({n_out/len(X_train)*100:.1f} %)')
print(f'Límites IQR: [{lower:.2f},  {upper:.2f}]')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].boxplot(X_train[col].dropna())
axes[0].set_title(f'Boxplot — {col}')

axes[1].hist(X_train[col].dropna(), bins=40, edgecolor='white', color='steelblue')
axes[1].axvline(lower, color='red', linestyle='--', label=f'Lower: {lower:.1f}')
axes[1].axvline(upper, color='red', linestyle='--', label=f'Upper: {upper:.1f}')
axes[1].legend()
axes[1].set_title(f'Histograma con límites IQR — {col}')

plt.tight_layout()
plt.show()

# Winsorización: capa sin eliminar filas (mismos límites de train aplicados a test)
# X_train[col] = X_train[col].clip(lower=lower, upper=upper)
# X_test[col]  = X_test[col].clip(lower=lower, upper=upper)

# Eliminación: siempre la misma máscara en X e y
# mask = (X_train[col] >= lower) & (X_train[col] <= upper)
# X_train = X_train[mask].reset_index(drop=True)
# y_train = y_train[mask].reset_index(drop=True)

### 6.6 Binning (Discretización)

Convierte una variable continua en categorías. Útil cuando la relación con la target es escalonada (no lineal) o cuando quieres controlar el efecto de outliers extremos sin eliminarlos.

**Cuidado**: se pierde información al discretizar. Usar solo cuando aporta valor real o simplifica la interpretación.

In [ ]:
# Binning manual con pd.cut (tú defines los intervalos)
# X_train['edad_grupo'] = pd.cut(
#     X_train['edad'],
#     bins=[0, 25, 45, 65, 100],
#     labels=['joven', 'adulto', 'senior', 'mayor']
# )

# Binning automático con cuantiles — misma cantidad de datos en cada bin
# X_train['precio_grupo'] = pd.qcut(X_train['precio'], q=4, labels=['Q1','Q2','Q3','Q4'])

# ⚠️ Los cortes deben calcularse en train y aplicarse en test con los mismos valores:
# bins_train = pd.cut(X_train['precio'], bins=4, retbins=True)[1]
# X_test['precio_grupo'] = pd.cut(X_test['precio'], bins=bins_train, labels=False)

### 6.7 Transformaciones de distribución

Las transformaciones reducen la **asimetría (skewness)** de una distribución para que se parezca más a una normal.

**¿Por qué importa?**

Modelos como regresión lineal/logística, SVM o PCA asumen (implícita o explícitamente) que las features tienen distribuciones razonablemente simétricas. Con distribuciones muy sesgadas, un outlier en la cola puede distorsionar el modelo. Los **árboles y ensembles no necesitan estas transformaciones**.

**¿Cuándo transformar?**

| Skewness (abs) | Interpretación | Transformación recomendada |
|---|---|---|
| < 0.5 | Distribución razonablemente simétrica | No necesaria |
| 0.5 – 1.0 | Asimetría moderada | `sqrt` |
| > 1.0 | Asimetría alta | `log1p` |
| Con valores negativos | `log` no aplica | `cbrt` |

> 💡 **¿Por qué `log1p` y no `log`?**  
> `log(0) = -∞` → el código explota si hay algún cero. `log1p(x) = log(x + 1)` evita ese problema y es prácticamente equivalente para valores grandes.  
> Lo mismo con `sqrt`: si puede haber negativos, usa `cbrt` (raíz cúbica), que funciona para cualquier valor real.

In [ ]:
# ── DEMO con datos sintéticos: ve el efecto antes de aplicarlo al dataset real

np.random.seed(42)
demo = np.random.exponential(scale=1.0, size=1000) * 50000 + 20000  # distribución tipo ingresos

transformaciones = {
    'Original':   demo,
    'log1p':      np.log1p(demo),
    'sqrt':       np.sqrt(np.clip(demo, 0, None)),
    'cbrt':       np.cbrt(demo),
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (nombre, datos) in zip(axes, transformaciones.items()):
    ax.hist(datos, bins=40, edgecolor='white', color='steelblue')
    skew_val = float(pd.Series(datos).skew())
    ax.set_title(f'{nombre}\nskew = {skew_val:.2f}', fontsize=11)

plt.suptitle('Efecto de cada transformación sobre la distribución', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── APLICAR al dataset real

col = 'feature'  # ← Cambiar

print(f'Skewness antes: {X_train[col].skew():.3f}')

# Aplicar la transformación elegida en train y en test por igual
X_train[col] = np.log1p(X_train[col])
X_test[col]  = np.log1p(X_test[col])   # misma transformación, no se recalcula

print(f'Skewness después de log1p: {X_train[col].skew():.3f}')

# Antes / después visual
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(np.expm1(X_train[col]), bins=40, edgecolor='white', color='salmon')
axes[0].set_title('Antes (escala original)')
axes[1].hist(X_train[col], bins=40, edgecolor='white', color='steelblue')
axes[1].set_title('Después (log1p)')
plt.tight_layout()
plt.show()

### 6.8 Encodings (variables categóricas)

Los modelos no entienden texto. Hay que convertirlo a números.

| Tipo de variable | Encoding | Ejemplo |
|---|---|---|
| Binaria | Map 0 / 1 | `sí/no` → `1/0` |
| Ordinal (tiene orden natural) | Map respetando el orden | `bajo/medio/alto` → `0/1/2` |
| Nominal sin orden | OneHot / get_dummies | `ciudad` → una columna por ciudad |
| Alta cardinalidad (> 15–20 cats) | Target Encoding, Hashing | `código_postal` |

> ⚠️ **Dummy trap**: OneHot con N categorías genera N columnas, pero la última es redundante (si no es ninguna de las N-1 primeras, es esa). Esto crea multicolinealidad perfecta. Solución: `drop='first'` en OneHotEncoder o `drop_first=True` en `pd.get_dummies()`.

> ⚠️ **Codificar siempre aprendiendo de train**. Si una categoría aparece en test pero no en train, el encoder debe manejarla (`handle_unknown='ignore'`).

In [ ]:
# Encoding binario manual
X_train['activo'] = X_train['activo'].map({'sí': 1, 'no': 0})
X_test['activo']  = X_test['activo'].map({'sí': 1, 'no': 0})

# OneHot Encoding — fit solo sobre train
cat_cols = X_train.select_dtypes('object').columns.tolist()

encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

ohe_train = encoder.fit_transform(X_train[cat_cols])  # fit + transform
ohe_test  = encoder.transform(X_test[cat_cols])        # solo transform

ohe_cols = encoder.get_feature_names_out(cat_cols)

X_train = pd.concat([
    X_train.drop(columns=cat_cols),
    pd.DataFrame(ohe_train, columns=ohe_cols, index=X_train.index)
], axis=1)

X_test = pd.concat([
    X_test.drop(columns=cat_cols),
    pd.DataFrame(ohe_test, columns=ohe_cols, index=X_test.index)
], axis=1)

### 6.9 Escalado

El escalado pone todas las variables en una escala comparable. Sin él, variables de rangos muy distintos dominan el modelo.

**¿Por qué importa?**  
En modelos basados en **gradiente** (regresión lineal/logística, redes neuronales), si una feature tiene valores en miles y otra en décimas, el gradiente se mueve a velocidades muy distintas y el entrenamiento es lento o inestable. En modelos basados en **distancias** (KNN, SVM), la distancia estará dominada por la variable con mayor escala aunque sea la menos informativa.

**Los árboles de decisión y Random Forest no necesitan escalado**, ya que sus decisiones son umbrales relativos, no distancias ni gradientes.

**Los tres scalers:**

| Scaler | Cómo funciona | Sensible a outliers | Cuándo usarlo |
|---|---|---|---|
| `StandardScaler` | Resta media, divide por std → media=0, std=1 | Sí | Distribuciones aproximadamente normales |
| `MinMaxScaler` | Mapea al rango [0, 1] | Mucho | Cuando necesitas un rango fijo; sin outliers |
| `RobustScaler` | Resta mediana, divide por IQR | No | Cuando hay outliers que no quieres eliminar |

> 💡 **¿Por qué RobustScaler es "robusto"?** Usa mediana e IQR, que son estadísticos resistentes a los extremos. Si tienes 99 sueldos de 25.000–35.000 € y uno de 250.000 €, StandardScaler distorsiona el escalado de todos; RobustScaler apenas lo nota.

In [ ]:
# ── DEMO: cómo afecta un outlier a cada scaler

np.random.seed(42)
datos_normales = np.random.normal(loc=30000, scale=5000, size=99)
outlier        = np.array([250000])   # un directivo
demo_salary    = np.concatenate([datos_normales, outlier]).reshape(-1, 1)

scalers = {
    'Original':      None,
    'StandardScaler': StandardScaler(),
    'MinMaxScaler':   MinMaxScaler(),
    'RobustScaler':   RobustScaler(),
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (nombre, scaler) in zip(axes, scalers.items()):
    scaled = demo_salary if scaler is None else scaler.fit_transform(demo_salary)
    ax.hist(scaled, bins=30, edgecolor='white', color='steelblue')
    ax.set_title(f'{nombre}\nmedia: {scaled.mean():.2f}  std: {scaled.std():.2f}', fontsize=10)

plt.suptitle('Efecto de cada scaler con un outlier extremo (sueldo 250k)', y=1.03, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── APLICAR al dataset real

num_cols = X_train.select_dtypes('number').columns.tolist()

scaler = StandardScaler()   # ← Cambiar según la situación

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])  # fit + transform
X_test[num_cols]  = scaler.transform(X_test[num_cols])        # solo transform

## Paso 7: Elección de métricas

La métrica mide cuán lejos están las predicciones del modelo de la realidad. Hay que elegirla **antes de entrenar**, según el tipo de problema y el coste de cada tipo de error.

### 7.1 Métricas para clasificación

#### Accuracy
Porcentaje total de aciertos. Fácil de entender pero engañosa con clases desbalanceadas: un modelo que siempre predice la clase mayoritaria puede tener 95 % de accuracy y ser inútil.

#### Matriz de confusión
La base de todo. Muestra exactamente dónde se equivoca el modelo.

```
                  Predicho
               Positivo  Negativo
Real Positivo |   TP   |   FN   |  ← FN: lo hay pero no lo detecta
     Negativo |   FP   |   TN   |  ← FP: lo predice pero no lo hay
```

#### Recall (Sensibilidad)
`TP / (TP + FN)` — De todos los positivos reales, ¿cuántos detecta el modelo?

Priorizar cuando un **falso negativo es el peor error**.  
Ejemplo: detección de enfermedades. Es peor decirle a alguien que está sano cuando tiene cáncer (FN), que hacerle pruebas innecesarias (FP).

#### Precision
`TP / (TP + FP)` — De todos los que predice como positivos, ¿cuántos son realmente positivos?

Priorizar cuando un **falso positivo es el peor error**.  
Ejemplo: filtro de spam. Es peor perder un email importante (FP), que dejar pasar algún spam (FN).

#### F1-Score
`2 · (Precision · Recall) / (Precision + Recall)` — Media armónica de Precision y Recall.

No es simplemente `(P + R) / 2`. La media armónica penaliza fuerte cuando uno de los dos es muy bajo. Si Precision = 1.0 y Recall = 0.1, F1 = 0.18 (no 0.55). Úsalo cuando ninguno puede sacrificarse completamente.

#### AUC-ROC
Evalúa el clasificador a **todos los posibles umbrales**, no solo 0.5. AUC = 1 → perfecto; AUC = 0.5 → aleatorio. Útil cuando quieres evaluar el modelo independientemente del umbral de decisión.

#### ¿Cómo elegir?

| Situación | Métrica principal |
|---|---|
| Clases balanceadas y coste simétrico | Accuracy o F1 |
| FN es más caro (medicina, fraude) | Recall |
| FP es más caro (spam, alertas) | Precision |
| Quiero evaluar sin fijar umbral | AUC-ROC |
| Clases muy desbalanceadas | F1, PR-AUC |

### 7.2 Métricas para regresión

| Métrica | Idea | Sensible a outliers | Interpretable |
|---|---|---|---|
| **MAE** | Media del \|error\| | Poco | Sí (mismas unidades que target) |
| **RMSE** | √(Media del error²) | Sí | Sí (mismas unidades) |
| **MSE** | Media del error² | Mucho | No (unidades²) |
| **R²** | % varianza explicada | Moderado | Sí (rango 0–1) |
| **MAPE** | % error medio absoluto | Sí si target ≈ 0 | Sí (%) |

> 💡 **RMSE vs MAE**: RMSE penaliza más los errores grandes (por el cuadrado). Si un error de 100 es mucho peor que 10 errores de 10, usa RMSE. Si todos los errores son igual de importantes, usa MAE.

> 💡 **R² no dice si el modelo es bueno en términos absolutos**: un R² = 0.7 puede ser excelente prediciendo precios de casas y pésimo en una predicción médica. Interpreta siempre en contexto.

> ⚠️ `mean_squared_error(..., squared=False)` fue eliminado en sklearn >= 1.4. Usa `root_mean_squared_error` directamente.

## Paso 8: Selección del modelo

No existe un modelo universalmente mejor. La elección depende de los datos y el objetivo.

| Factor | Modelos simples | Modelos complejos |
|---|---|---|
| **Volumen de datos** | Pocos datos | Muchos datos |
| **Interpretabilidad** | Alta (regresiones, DT) | Baja (ensembles, redes) |
| **Velocidad de entrenamiento** | Alta | Baja |
| **Tipo de relación** | Lineal | No lineal |

> 💡 **Entrena siempre un baseline primero**. Un modelo simple (regresión lineal, árbol superficial) te da un punto de referencia. Si un modelo complejo no lo mejora significativamente, no vale la pena su complejidad adicional.

**Precision vs interpretabilidad**  
Los modelos interpretables ("caja blanca") permiten explicar cada decisión: regresiones, árboles simples.  
Los modelos complejos ("caja negra") suelen ser más precisos pero difíciles de justificar: Random Forest, Gradient Boosting, redes neuronales.  
La elección depende de si el negocio necesita entender y auditar las decisiones del modelo.

## Paso 9: Entrenamiento del modelo

El modelo aprende patrones a partir de los datos de entrenamiento. El objetivo no es que funcione bien en train, sino que **generalice a datos nuevos**.

Antes de entrenar:
- Split correcto, preprocesado solo con train.
- Métrica de evaluación definida.
- Modelo base (baseline) para tener referencia.

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

## Paso 10: Hiperparametrización y Regularización

### 10.1 GridSearchCV

Los hiperparámetros no se aprenden del entrenamiento: hay que definirlos antes. Como no podemos saber cuáles son los mejores a priori, probamos combinaciones de forma automática.

> ⚠️ El GridSearch **solo debe entrenarse con train** (usa validación cruzada interna). Usar test para elegir hiperparámetros también es data leakage.

In [ ]:
# ── Grids de hiperparámetros por modelo

# REGRESION LOGISTICA
grid_logreg = {
    'penalty':  ['l1', 'l2'],
    'C':        [0.01, 0.1, 0.5, 1.0, 5.0, 10.0],
    'max_iter': [100, 500],
    'solver':   ['liblinear']
}

# ARBOL DE DECISION
grid_arbol = {
    'max_depth':         list(range(1, 10)),
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 5]
}

# RANDOM FOREST
grid_rf = {
    'n_estimators': [100, 150],       # A partir de ~100 la mejora es marginal
    'max_depth':    [3, 5, 10, 15, None],
    'max_features': ['sqrt', 0.3, 0.5]
}

# GRADIENT BOOSTING
grid_gb = {
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators':  [50, 100, 200],   # Cuidado: más árboles → más riesgo de overfitting
    'max_depth':     [2, 3, 4]         # Árboles poco profundos bastan
}

In [ ]:
# ── Ejemplo de uso

model = LogisticRegression(solver='liblinear')

grid = GridSearchCV(
    estimator=model,
    param_grid=grid_logreg,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print('Mejores hiperparámetros:', grid.best_params_)
print('Mejor score en CV:',       round(grid.best_score_, 3))

best_model = grid.best_estimator_

### 10.2 Regularización: controlando el sobreajuste

La regularización añade una **penalización** a la función de coste para evitar que los coeficientes crezcan sin control. Un modelo sin regularización puede memorizarse el ruido del train (overfitting) y fallar en datos nuevos.

---

#### L2 — Ridge
Penaliza la **suma de los cuadrados** de los coeficientes.

Geométricamente: fuerza a los coeficientes a quedarse dentro de una **esfera**. El punto óptimo tiende a caer sobre la superficie de la esfera, donde todos los coeficientes se **reducen** pero **ninguno llega exactamente a cero**. Útil cuando todas las features son relevantes y quieres estabilizar el modelo.

#### L1 — Lasso
Penaliza la **suma del valor absoluto** de los coeficientes.

Geométricamente: la restricción es un **diamante** (rombo). Sus vértices están exactamente sobre los ejes. El punto óptimo tiende a caer en un vértice, forzando uno o más coeficientes a ser **exactamente cero**. Esto equivale a eliminar features: **L1 hace selección de variables de forma implícita**.

#### ElasticNet
Combina L1 y L2. Útil cuando hay muchas features correlacionadas (L1 tiende a quedarse solo con una de cada grupo; ElasticNet puede conservar varias).

---

> ⚠️ **Confusión muy habitual: `C` vs `alpha`**  
> En `Ridge` y `Lasso`: `alpha` alto → más regularización → coeficientes más pequeños.  
> En `LogisticRegression`: el parámetro es `C` y funciona **al revés**. `C` alto → menos regularización.  
> La relación es `C = 1 / alpha`. Es un detalle de sklearn que confunde mucho al principio.

In [ ]:
# ── DEMO: rutas de coeficientes con Ridge vs Lasso
# Generamos datos con 10 features, de las cuales solo 4 son informativas

from sklearn.datasets import make_regression

X_demo, y_demo = make_regression(
    n_samples=200, n_features=10, n_informative=4,
    noise=20, random_state=42
)

alphas = np.logspace(-3, 3, 60)

ridge_coefs = [Ridge(alpha=a).fit(X_demo, y_demo).coef_ for a in alphas]
lasso_coefs = [Lasso(alpha=a, max_iter=5000).fit(X_demo, y_demo).coef_ for a in alphas]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

titulos = [
    'Ridge (L2): coeficientes se reducen pero nunca llegan a 0',
    'Lasso (L1): coeficientes llegan a 0 → selección automática de features'
]

for coef_paths, ax, titulo in zip([ridge_coefs, lasso_coefs], axes, titulos):
    for i, coef_series in enumerate(np.array(coef_paths).T):
        ax.plot(alphas, coef_series, label=f'feature {i}')
    ax.set_xscale('log')
    ax.invert_xaxis()
    ax.axhline(0, color='black', linestyle='--', linewidth=0.7)
    ax.set_xlabel('alpha  (← menos regularización | más regularización →)')
    ax.set_ylabel('Valor del coeficiente')
    ax.set_title(titulo, fontsize=10)
    ax.legend(loc='upper left', fontsize=7, ncol=2)

plt.tight_layout()
plt.show()

# ¿Qué features elimina Lasso con alpha=1?
lasso_fit = Lasso(alpha=1.0).fit(X_demo, y_demo)
print('\nCoeficientes Lasso (alpha=1):')
for i, c in enumerate(lasso_fit.coef_):
    estado = '✓ activa' if abs(c) > 1e-4 else '✗ eliminada (coef = 0)'
    print(f'  feature {i:2d}: {c:8.3f}  {estado}')

In [ ]:
# ── APLICAR al dataset real

ridge = Ridge(alpha=1.0)
lasso = Lasso(alpha=0.1)
en    = ElasticNet(alpha=0.1, l1_ratio=0.5)

for nombre, modelo in [('Ridge', ridge), ('Lasso', lasso), ('ElasticNet', en)]:
    modelo.fit(X_train, y_train)
    rmse = root_mean_squared_error(y_test, modelo.predict(X_test))
    print(f'{nombre:12s}  RMSE test: {rmse:.3f}')

## Paso 11: Evaluación

Evaluamos el modelo final sobre test. Es la primera (y única) vez que usamos test para tomar una decisión.

### Overfitting vs Underfitting

```
                     Train       Test
  Underfitting:      malo        malo       → modelo demasiado simple
  Buen ajuste:       bueno       bueno      → generaliza
  Overfitting:       muy bueno   malo       → memorizó el train
```

**Si hay overfitting:**
- Aumentar regularización (`alpha` en Ridge/Lasso, reducir `C` en LogReg, `max_depth` en árboles).
- Conseguir más datos de entrenamiento.
- Eliminar features irrelevantes (PCA, feature selection).
- Usar Cross-Validation como técnica de evaluación más robusta.

**Si hay underfitting:**
- Probar modelos más complejos.
- Añadir más features o transformaciones.
- Reducir regularización.

### Evaluación para regresión

In [ ]:
y_train_pred = best_model.predict(X_train)
y_test_pred  = best_model.predict(X_test)

print(f'Train RMSE: {root_mean_squared_error(y_train, y_train_pred):.3f}')
print(f'Test  RMSE: {root_mean_squared_error(y_test,  y_test_pred):.3f}')
print(f'Train R²:   {r2_score(y_train, y_train_pred):.3f}')
print(f'Test  R²:   {r2_score(y_test,  y_test_pred):.3f}')
print()

# Cross-validation: evaluación más robusta usando solo train
cv_r2 = cross_val_score(best_model, X_train, y_train, cv=5, scoring='r2')
print(f'CV R² (media ± std): {cv_r2.mean():.3f} ± {cv_r2.std():.3f}')

# Residuos: deben ser aleatorios, sin patrón
residuos = y_test - y_test_pred
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(y_test_pred, residuos, alpha=0.4, s=15)
ax.axhline(0, color='red', linestyle='--')
ax.set_xlabel('Predicciones')
ax.set_ylabel('Residuos')
ax.set_title('Residuos vs predicciones — si hay patrón, el modelo tiene sesgo')
plt.tight_layout()
plt.show()

### Evaluación para clasificación

In [ ]:
y_train_pred = best_model.predict(X_train)
y_test_pred  = best_model.predict(X_test)

print(f'Train Accuracy:  {accuracy_score(y_train, y_train_pred):.3f}')
print(f'Test  Accuracy:  {accuracy_score(y_test,  y_test_pred):.3f}')
print(f'Test  Precision: {precision_score(y_test, y_test_pred, average="weighted"):.3f}')
print(f'Test  Recall:    {recall_score(y_test, y_test_pred, average="weighted"):.3f}')
print(f'Test  F1-score:  {f1_score(y_test, y_test_pred, average="weighted"):.3f}')
print()

# Matriz de confusión
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred, ax=ax, colorbar=False)
ax.set_title('Matriz de confusión')
plt.tight_layout()
plt.show()

# Cross-validation
cv_f1 = cross_val_score(best_model, X_train, y_train, cv=5, scoring='f1_weighted')
print(f'CV F1 (media ± std): {cv_f1.mean():.3f} ± {cv_f1.std():.3f}')

## Paso 12: Conclusión

Las conclusiones deben responder a estos puntos de forma clara y orientada al negocio:

1. **Rendimiento general** — Métricas clave con interpretación concreta (no solo el número).
2. **Errores y limitaciones** — Dónde falla más y por qué. Posibles sesgos.
3. **Importancia de variables** — Qué features fueron más determinantes.
4. **Aplicabilidad** — Cómo ayudan los resultados a la toma de decisiones. Cuándo no usar el modelo.
5. **Recomendaciones futuras** — Más datos, refinamiento de features, otros modelos, monitoreo en producción.

**Ejemplo de redacción para presentar resultados:**

> *El modelo de clasificación logra un 82 % de accuracy y un F1-score de 0.77, lo que indica un buen equilibrio entre precisión y recall. Detecta correctamente la mayoría de los casos positivos, aunque confunde principalmente X con Y. Las variables más relevantes fueron A, B y C, lo que concuerda con el conocimiento del negocio. Se recomienda seguir recopilando datos y evaluar el rendimiento regularmente para detectar degradación del modelo a lo largo del tiempo.*